# Async-input distillation: one-example sanity check

A Qwen3.5-4B teacher starts from shard 1, receives shard 2 while reasoning, and records the executed cache trace. Qwen3.5-0.8B overfits that trace, reloads its checkpoint, then repeats the async-input protocol independently.

In [ ]:
import json, os, sys, tempfile
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/async_input_distillation_sanity_check').is_dir())
sys.path.insert(0, str(ROOT))
from transformers import AutoTokenizer
from scripts.async_input_distillation_sanity_check.sanity import check_answer, check_async_behavior, compare_rollouts, fit_student, load_example, record_rollout, verify_checkpoint
GPU, PRECISION, OPTIMIZER_FOREACH = '0', 'float16', False
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
TEACHER, STUDENT = 'Qwen/Qwen3.5-4B', 'Qwen/Qwen3.5-0.8B'
DATASET, SPLIT, INDEX = 'MrDarkTesla/math_500_sharded', 'train', 3
K_STEPS, MAX_TOKENS = 16, 512
STEPS, LR = 100, 1e-6
runs = ROOT / 'training_runs/async_input_distillation_sanity_check'
runs.mkdir(parents=True, exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix='math500_sharded_', dir=runs))
tokenizer = AutoTokenizer.from_pretrained(STUDENT)
print('Run directory:', RUN)

## Record and validate one teacher trajectory

In [ ]:
teacher_trace = record_rollout(ROOT, GPU, TEACHER, RUN / 'teacher', dataset=DATASET, split=SPLIT, index=INDEX, k_steps=K_STEPS, max_tokens=MAX_TOKENS, precision=PRECISION)
teacher = load_example(teacher_trace)
print('Shard 1:', teacher['problem_shards'][0])
print('Correction:', teacher['problem_shards'][1])
print('Output:', tokenizer.decode(teacher['output_tokens'], skip_special_tokens=True))
teacher_answer = check_answer(teacher, tokenizer, teacher['answer'])
teacher_async = check_async_behavior(teacher, tokenizer)
assert teacher_answer['status'] == 'PASS', teacher_answer
assert teacher_async['status'] == 'PASS', teacher_async

## Overfit the student on the recorded targets

In [ ]:
metrics = fit_student(ROOT, GPU, teacher_trace, STUDENT, RUN / 'trained', steps=STEPS, lr=LR, precision=PRECISION, optimizer_foreach=OPTIMIZER_FOREACH)
print(metrics['initial'], '->', metrics['final'])
assert metrics['overfit_achieved'], metrics
checkpoint = Path(metrics['checkpoint'])

## Reload the checkpoint and verify teacher-forced replay

In [ ]:
reloaded = verify_checkpoint(ROOT, GPU, teacher_trace, checkpoint, RUN / 'reloaded.json', precision=PRECISION)
assert reloaded['accuracy'] == 1.0 and reloaded['loss'] < 0.01, reloaded
reloaded

## Run the trained student independently with the same delayed correction

In [ ]:
student_trace = record_rollout(ROOT, GPU, checkpoint, RUN / 'student', dataset=DATASET, split=SPLIT, index=INDEX, k_steps=K_STEPS, max_tokens=MAX_TOKENS, precision=PRECISION)
student = load_example(student_trace)
print(tokenizer.decode(student['output_tokens'], skip_special_tokens=True))
report = compare_rollouts(teacher_trace, student_trace, tokenizer, teacher['answer'], RUN / 'checks.json')
for name, result in report.items(): print(name, result['status'], '-', result['reason'])
print('Full report:', RUN / 'checks.json')